# Triangles vs quads — 2D conformal mesh

Same grain polygons, two `mesh_gs` calls: linear triangles (`recombine_to_quads=False`)
and quads (`recombine_to_quads=True`, `mesh_algo=8`). Plot and Abaqus INP for both.

Requires `gmsh` (`pip install upxo[mesh]`). `confMesh2d` (pygmsh) is deprecated.
Canonical mesh-only demo: `confMesh2d_gmsh.ipynb`.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import box

from upxo.meshing.gsmesh2d import mesh_gs
from upxo.meshing.writer_ABQ import summarize_inp


In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
}

`mesh_algo=6` (Frontal) for triangles; `mesh_algo=8` (Frontal-Delaunay for quads) when recombining.

In [ ]:
tri = mesh_gs(cells, mesh_size_gb=0.35, mesh_size_bulk=0.7,
             mesh_algo=6, recombine_to_quads=False)
quad = mesh_gs(cells, mesh_size_gb=0.35, mesh_size_bulk=0.7,
              mesh_algo=8, recombine_to_quads=True)
mt, mq = tri['mesher'], quad['mesher']
for name, r, m in (('tri', tri, mt), ('quad', quad, mq)):
    m.form_elsets_gmsh(); m.build_boundary_nsets(); m.build_gb_nset()
    print(name, 'n_tri', r['n_tri'], 'n_quad', r['n_quad'], m.validation_report)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), dpi=120)
mt.plot_by_grain(ax=axes[0], show_gb=True, show_nsets=False, title='triangles')
mq.plot_by_grain(ax=axes[1], show_gb=True, show_nsets=False, title='quads')
fig.tight_layout()
fig

In [ ]:
out = Path.cwd() / 'confMesh2d_tri_vs_quad_out'
p_tri = mt.export_abaqus_inp(out / 'rve_cps3.inp', plane='stress')
p_quad = mq.export_abaqus_inp(out / 'rve_cps4.inp', plane='stress')
p_tri, summarize_inp(p_tri), p_quad, summarize_inp(p_quad)